# Exercise 2 - Perturbation Modeling

Transition objective: predict next cell state with `[X_t, P_t] -> X_{t+1}`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path.cwd() / 'src'))

from perturbation_pipeline import (
    compute_umap_embeddings,
    evaluate_predictions_by_timepoint,
    load_experiment_data,
    merge_train_val_splits,
    plot_umap_by_timepoint,
    split_dataset,
    test,
    train_with_model_selection,
)

In [2]:
DATA_DIR = Path('simulated_data_for_interview_exercise')
adata = load_experiment_data(DATA_DIR)

print('adata shape:', adata.shape)
print('expression dim:', adata.n_vars)
print('perturbation dim:', adata.obsm['perturbation'].shape[1])
print('timepoints:', sorted(adata.obs['round'].unique().tolist()))

adata shape: (48070, 50)
expression dim: 50
perturbation dim: 8
timepoints: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [ ]:
adata = compute_umap_embeddings(adata)
fig = plot_umap_by_timepoint(adata, save_path='umap_timepoint.png')
fig

adata.obsm["perturbation"]: many identical rows.

- Total rows: 48,070
- Unique rows: 9,908
- Extra duplicate rows: 38,162

Per round, perturbations are also heavily repeated (~989–994 unique for 4,807 rows each round).

Across rounds, almost all unique perturbation vectors are round-specific, except one vector shared across all 10 rounds: the all-zero perturbation (appears 186 times total).

In [3]:
splits = split_dataset(
    adata=adata,
    test_timepoints=(9, 10),
    val_fraction=0.2,
    random_state=42,
)

for split_name in ['train', 'val', 'test']:
    x_split, y_split = splits[split_name]
    print(split_name, 'X:', x_split.shape, 'y:', y_split.shape)

train X: (26920, 58) y: (26920, 50)
val X: (6729, 58) y: (6729, 50)
test X: (9614, 58) y: (9614, 50)


In [ ]:
# Robust training: model selection with k-fold CV on train split
train_pool = merge_train_val_splits(splits)

selected = train_with_model_selection(
    train_data=train_pool,
    candidate_models=('mlp', 'linear_regression', 'xgboost'),
    n_splits=5,
    random_state=42,
    mlp_hidden_dim=128,
    mlp_max_iter=300,
)

print('Best model:', selected['best_model_name'])

summary_rows = []
for model_name, result in selected['cv_results'].items():
    row = {'model': model_name}
    row.update({f'mean_{k}': v for k, v in result['mean_metrics'].items()})
    row.update({f'std_{k}': v for k, v in result['std_metrics'].items()})
    summary_rows.append(row)

pd.DataFrame(summary_rows).sort_values('mean_rmse').reset_index(drop=True)

In [ ]:
test_output = test(selected['model'], splits['test'])
print('Test metrics:', test_output['metrics'])

per_tp_metrics = evaluate_predictions_by_timepoint(
    prediction=test_output['prediction'],
    ground_truth=splits['test'][1],
    timepoints=splits['timepoints']['test'],
)
per_tp_metrics
